In [43]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
import sqlite3

In [61]:
headers = {
    'User-Agent' : 'Mozilla/5.0'
}

base_url = 'https://quotes.toscrape.com/'

s = requests.Session()
s.headers.update(headers)

def Fetch(remaing_url):
    final_url = base_url+remaing_url
    try:
        web = s.get(final_url, timeout=15)
        web.raise_for_status()
        return web
    except requests.RequestException as f:
        print(f)
        return None

def Make_Soup(web):
    soup = BeautifulSoup(web.content, 'html.parser')
    return soup


data_list = []

# Creating/Intializing the table scheema ---
# Create_Table()
# Delete_Table()

for i in range(1, 11):
    url = f'page/{i}/'
    web=Fetch(url)
    if web:
        soup=Make_Soup(web)
        coutes=soup.find_all('div', class_='quote')
        for coute in coutes:
            text=coute.find('span', class_='text').text
            author=coute.find('small', class_='author').text
            diff_tags=coute.find_all('a', class_='tag')
            tags = [tag.text for tag in diff_tags]
            # for tag in diff_tags:
            #     tags.append(tag.text)
            data_list.append({
                'Coute' : text,
                'Author' : author,
                'Tags' : tags
            })
            
            # Insert data into Database ---
            Insert(text, author)
            
    delay=random.uniform(1,5)
    time.sleep(delay)
    print(f'The script wait for - {delay:.2F} sec')
df=pd.DataFrame(data_list)

# Clossing the database connections ---
Clossing_conn()
print('all are Done')

The script wait for - 2.88 sec
The script wait for - 3.98 sec
The script wait for - 4.97 sec
The script wait for - 4.46 sec
The script wait for - 3.24 sec
The script wait for - 1.64 sec
The script wait for - 3.61 sec
The script wait for - 2.26 sec
The script wait for - 4.32 sec
The script wait for - 1.02 sec
connection are closed
all are Done


In [30]:
df.to_csv('Quotes_web_scrape.csv', index=False)

In [8]:
df.head(10)

,Coute,Author,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]"
5,“Try not to become a man of success. Rather be...,Albert Einstein,"[adulthood, success, value]"
6,“It is better to be hated for what you are tha...,André Gide,"[life, love]"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"[edison, failure, inspirational, paraphrased]"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,[misattributed-eleanor-roosevelt]
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"[humor, obvious, simile]"


In [67]:
connection=sqlite3.connect('Quotes.db')
cursor=connection.cursor()

In [73]:
# Creating Table scheema ---
def Create_Table():
    cursor.execute('''
        CREATE TABLE quotes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            quote VARCHAR(200),
            author VARCHAR(40)
        );
    ''')
    connection.commit()

# Inserting data Into the Table ----
def Insert(quote, author):

    cursor.execute(f'''
        INSERT INTO quotes (quote, author) VALUES(?, ?);
    ''', (str(quote), str(author)))
    connection.commit()

# Clossing the connection ---
def Clossing_conn():
    cursor.close()
    connection.close()
    print('connection are closed')

# Drop the Table ---
def Delete_Table():
    cursor.execute('''DROP TABLE quotes; ''')
    connection.commit()
    print('table was droped done')

# Fetch data from the table ---
def Fetch_data(table):
    cursor=connection.cursor()
    cursor.execute(f'''
        SELECT * FROM {table};
    ''')
    data=cursor.fetchall();
    return data

In [51]:
Clossing_conn()

connection are closed


In [76]:
data = Fetch_data('quotes')
for row in data:
    print(row)
print(len(data))

(1, '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'Albert Einstein')
(2, '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'J.K. Rowling')
(3, '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'Albert Einstein')
(4, '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', 'Jane Austen')
(5, "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", 'Marilyn Monroe')
(6, '“Try not to become a man of success. Rather become a man of value.”', 'Albert Einstein')
(7, '“It is better to be hated for what you are than to be loved for what you are not.”', 'André Gide')
(8, "“I have not failed. I've just found 10,000 ways that won't work.”", 'Thomas A. Edison')
(9, "“A woman is like a tea bag; you